# Query a Fabric Data Agent through MCP

Connect directly to the published Fabric Data Agent MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The endpoint uses the signed-in Azure Developer CLI identity and requires access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [14]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_DATA_AGENT_MCP_URL = os.environ["FABRIC_DATA_AGENT_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"


@asynccontextmanager
async def open_data_agent_session():
    credential = AzureDeveloperCliCredential(tenant_id=FABRIC_TENANT_ID)
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_DATA_AGENT_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric Data Agent configuration loaded")

Fabric Data Agent configuration loaded


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the published data agent, including each tool's input schema.

In [12]:
from rich.console import Console

console = Console()

async with open_data_agent_session() as session:
    tools_response = await session.list_tools()

data_agent_tools = tools_response.tools

for tool in data_agent_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

Tool: DataAgent_ContosoDIYDataAgent

Description: Contoso DIY product operations and review intelligence data agent

Input schema:

{
  "type": "object",
  "properties": {
    "userQuestion": {
      "type": "string"
    }
  },
  "required": [
    "userQuestion"
  ]
}

## Step 3: Call the Fabric Data Agent tool

Use the first tool discovered above and derive its question argument from the advertised input schema. Change `question` to explore other product and inventory questions.

In [13]:
from rich.markdown import Markdown

question = "Which products currently have the lowest inventory quantities?"
tool = data_agent_tools[0]
properties = tool.inputSchema.get("properties", {})
if not properties:
    raise RuntimeError(f"MCP tool '{tool.name}' has no input properties.")

question_argument = next(iter(properties))
async with open_data_agent_session() as session:
    result = await session.call_tool(
        tool.name,
        {question_argument: question},
    )

answers = [block.text for block in result.content if block.type == "text"]
if not answers:
    raise RuntimeError(f"MCP tool '{tool.name}' returned no text content.")

console.print(Markdown("\n\n".join(answers)))

The following products currently have the lowest inventory quantities (available quantity is 0):                   

 • Security Hasp 6-inch (SKU: HWHL000036)                                                                          
 • Pole Saw Pruner 8-foot (SKU: GOPRN002)                                                                          
 • Clear Polyurethane Satin (SKU: PFPU000036)                                                                      
 • OSB Subflooring 4x8x3/4 (SKU: LBOSB002)                                                                         
 • Modular Cabinet System (SKU: SOCA000020)                                                                        
 • Heavy Duty Jigsaw (SKU: PTJS000019)                                                                             
 • Deck Post 6x6x10 (SKU: LBTRT005)                                                                                
 • All-Purpose Fertilizer 50lb (SKU: GOFRT001)                                                                     
 • Standard Outlet 15-Amp (SKU: ELOTL001)                                                                          

All of these products are currently out of stock at their respective store locations.